In [0]:
from pyspark.sql import Window, functions as F
from pyspark.sql.functions import col, current_timestamp, trim, regexp_replace, when, row_number, lit, coalesce, upper

In [0]:
STORAGE_ACCOUNT = "hantstorageaccount"
TRAINING_SOURCE_CONTAINER = "training-invoices"
LAKEHOUSE_CONTAINER = "lakehouse"

TRAINING_SOURCE_PATH = f"abfss://{TRAINING_SOURCE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/"

TRAINING_BRONZE_CSV_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/training/bronze/training_invoice_raw/"
TRAINING_SCHEMA_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/cloudfiles_schema/training_invoice/"
TRAINING_CHECKPOINT_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/checkpoints/training_invoice/"

TRAINING_SILVER_HEADER_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/training/silver/training_invoice_cleansed/header/"
TRAINING_SILVER_LINES_PATH = f"abfss://{LAKEHOUSE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/training/silver/training_invoice_cleansed/lines/"

In [0]:
invoices_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", TRAINING_SCHEMA_PATH)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(TRAINING_SOURCE_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingest_ts", current_timestamp())
)

(
    invoices_stream.writeStream
    .format("delta")
    .option("checkpointLocation", TRAINING_CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .start(TRAINING_BRONZE_CSV_PATH)
    .awaitTermination()
)

In [0]:
training_data_df = spark.read.format("delta").load(TRAINING_BRONZE_CSV_PATH)
print(f"Total records in training data: {training_data_df.count()}")
display(training_data_df)

Total records in training data: 6879


InvoiceId,OrderDate,CustomerName,ShipPostalCode,ShipCity,ShipState,ShipCountry,ShipMode,BalanceDue,SubTotal,DiscountPercent,DiscountAmount,ShippingAmount,InvoiceTotal,OrderId,ProductName,SubCategory,Category,ProductId,Quantity,UnitPrice,ItemSubTotal,_rescued_data,_source_file,_ingest_ts
31151,2024-10-16,Elina Virtanen,75216,Tampere,Pirkanmaa,Finland,Standard Class,3672.1,3661.07,0.0,0.0,11.03,3672.1,EV-2024-6242810-49421,"Smartphone, 256GB (Silver)",Phones,Technology,TEC-PH-2284,3.0,894.0446,2682.13,null,abfss://training-invoices@hantstorageaccount.dfs.core.windows.net/training_dataset_3.csv,2026-04-22T21:55:04.906Z
31152,2024-09-14,Mikko Laine,46373,Tampere,Pirkanmaa,Finland,Standard Class,1892.38,1880.65,0.0,0.0,11.73,1892.38,ML-2024-6753430-52375,"Archive Storage Box, Letter Size",Storage,Office Supplies,OFF-ST-6900,7.0,13.3065,93.15,null,abfss://training-invoices@hantstorageaccount.dfs.core.windows.net/training_dataset_3.csv,2026-04-22T21:55:04.906Z
31152,2024-09-14,Mikko Laine,46373,Tampere,Pirkanmaa,Finland,Standard Class,1892.38,1880.65,0.0,0.0,11.73,1892.38,ML-2024-6753430-52375,"Wireless Headset, Noise Cancelling",Accessories,Technology,TEC-AC-5512,4.0,137.0638,548.26,null,abfss://training-invoices@hantstorageaccount.dfs.core.windows.net/training_dataset_3.csv,2026-04-22T21:55:04.906Z
31152,2024-09-14,Mikko Laine,46373,Tampere,Pirkanmaa,Finland,Standard Class,1892.38,1880.65,0.0,0.0,11.73,1892.38,ML-2024-6753430-52375,"Laptop 14in, 16GB RAM",Computers,Technology,TEC-CO-2655,1.0,1239.2426,1239.24,null,abfss://training-invoices@hantstorageaccount.dfs.core.windows.net/training_dataset_3.csv,2026-04-22T21:55:04.906Z
31153,2023-02-25,Elias Virtala,40356,New York,New York,United States,Same Day,1285.21,1498.1,0.15,224.72,11.83,1285.21,EV-2023-5763303-90078,"Bookshelf, Industrial Metal/Wood",Bookcases,Furniture,FUR-BO-4578,5.0,251.2202,1256.1,null,abfss://training-invoices@hantstorageaccount.dfs.core.windows.net/training_dataset_3.csv,2026-04-22T21:55:04.906Z
31153,2023-02-25,Elias Virtala,40356,New York,New York,United States,Same Day,1285.21,1498.1,0.15,224.72,11.83,1285.21,EV-2023-5763303-90078,"Stapler, Heavy Duty",Fasteners,Office Supplies,OFF-FA-2801,8.0,24.4156,195.32,null,abfss://training-invoices@hantstorageaccount.dfs.core.windows.net/training_dataset_3.csv,2026-04-22T21:55:04.906Z
31153,2023-02-25,Elias Virtala,40356,New York,New York,United States,Same Day,1285.21,1498.1,0.15,224.72,11.83,1285.21,EV-2023-5763303-90078,"USB-C Charger 65W, GaN",Accessories,Technology,TEC-AC-7434,1.0,46.6821,46.68,null,abfss://training-invoices@hantstorageaccount.dfs.core.windows.net/training_dataset_3.csv,2026-04-22T21:55:04.906Z
31154,2023-04-24,Daniel Murphy,84524,Hamburg,Hamburg,Germany,Standard Class,7594.32,7972.75,0.05,398.64,20.21,7594.32,DM-2023-7983013-41449,"Bookshelf, Industrial Metal/Wood",Bookcases,Furniture,FUR-BO-4578,1.0,212.3321,212.33,null,abfss://training-invoices@hantstorageaccount.dfs.core.windows.net/training_dataset_3.csv,2026-04-22T21:55:04.906Z
31154,2023-04-24,Daniel Murphy,84524,Hamburg,Hamburg,Germany,Standard Class,7594.32,7972.75,0.05,398.64,20.21,7594.32,DM-2023-7983013-41449,"Laptop 14in, 16GB RAM",Computers,Technology,TEC-CO-2655,5.0,1552.0833,7760.42,null,abfss://training-invoices@hantstorageaccount.dfs.core.windows.net/training_dataset_3.csv,2026-04-22T21:55:04.906Z
31155,2025-07-28,Mikko Laine,85064,Valencia,Valencian Community,Spain,Same Day,1148.57,1326.67,0.15,199.0,20.9,1148.57,ML-2025-5465895-68653,"Modular Bookcase, Walnut (6-Cube)",Bookcases,Furniture,FUR-BO-5017,1.0,222.7635,222.76,null,abfss://training-invoices@hantstorageaccount.dfs.core.windows.net/training_dataset_3.csv,2026-04-22T21:55:04.906Z


In [0]:
def _clean_numeric_str(col_name):
    return regexp_replace(trim(col(col_name).cast("string")), r"[$,%\s,]", "")


def _safe_decimal(col_name):
    cleaned = _clean_numeric_str(col_name)
    return when((cleaned == "") | cleaned.isNull(), None).otherwise(cleaned.cast("decimal(18,2)"))


def _safe_percent(col_name):
    cleaned = _clean_numeric_str(col_name)
    return when((cleaned == "") | cleaned.isNull(), None).otherwise(
        when(cleaned.cast("double") > 1, (cleaned.cast("double") / 100.0)).otherwise(cleaned.cast("double")).cast("decimal(9,6)")
    )


def _safe_int(col_name):
    cleaned = regexp_replace(trim(col(col_name).cast("string")), r"[,\s]", "")
    return when((cleaned == "") | cleaned.isNull(), None).otherwise(cleaned.cast("double").cast("int"))


def _safe_date(col_name):
    raw = trim(col(col_name).cast("string"))
    return coalesce(
        F.try_to_date(raw, "yyyy-MM-dd"),
        F.try_to_date(raw, "M/d/yyyy"),
        F.try_to_date(raw, "MM/dd/yyyy"),
        F.try_to_date(raw, "d-MMM-yyyy"),
    )


def _trim_and_nullify(col_name):
    raw = trim(col(col_name).cast("string"))
    return when(raw == "", None).otherwise(raw)

def _upper_case_trim(col_name):
    return upper(trim(regexp_replace(col(col_name).cast("string"), r"\s+", " ")))

In [0]:
if training_data_df.rdd.isEmpty():
    raise RuntimeError("Data transform cannot start because raw data is empty.")

In [0]:
training_header_df = (
    training_data_df
    .withColumn("InvoiceId", _trim_and_nullify("InvoiceId"))
    .withColumn("OrderDate", _safe_date("OrderDate"))
    .withColumn("CustomerName", _trim_and_nullify("CustomerName"))
    .withColumn("ShipPostalCode", _trim_and_nullify("ShipPostalCode"))
    .withColumn("ShipCity", _trim_and_nullify("ShipCity"))
    .withColumn("ShipState", _trim_and_nullify("ShipState"))
    .withColumn("ShipCountry", _trim_and_nullify("ShipCountry"))
    .withColumn("ShipMode", _upper_case_trim("ShipMode"))
    .withColumn("BalanceDue", _safe_decimal("BalanceDue"))
    .withColumn("SubTotal", _safe_decimal("SubTotal"))
    .withColumn("DiscountPercent", _safe_percent("DiscountPercent"))
    .withColumn("DiscountAmount", _safe_decimal("DiscountAmount"))
    .withColumn("ShippingAmount", _safe_decimal("ShippingAmount"))
    .withColumn("InvoiceTotal", _safe_decimal("InvoiceTotal"))
    .withColumn("OrderId", _trim_and_nullify("OrderId"))
    .groupBy("InvoiceId")
    .agg(
        F.first("_source_file", ignorenulls=True).alias("_source_file"),
        F.first("OrderDate", ignorenulls=True).alias("OrderDate"),
        F.first("CustomerName", ignorenulls=True).alias("CustomerName"),
        F.first("ShipPostalCode", ignorenulls=True).alias("ShipPostalCode"),
        F.first("ShipCity", ignorenulls=True).alias("ShipCity"),
        F.first("ShipState", ignorenulls=True).alias("ShipState"),
        F.first("ShipCountry", ignorenulls=True).alias("ShipCountry"),
        F.first("ShipMode", ignorenulls=True).alias("ShipMode"),
        F.first("BalanceDue", ignorenulls=True).alias("BalanceDue"),
        F.first("SubTotal", ignorenulls=True).alias("SubTotal"),
        F.first("DiscountPercent", ignorenulls=True).alias("DiscountPercent"),
        F.first("DiscountAmount", ignorenulls=True).alias("DiscountAmount"),
        F.first("ShippingAmount", ignorenulls=True).alias("ShippingAmount"),
        F.first("InvoiceTotal", ignorenulls=True).alias("InvoiceTotal"),
        F.first("OrderId", ignorenulls=True).alias("OrderId"),
    )
    .withColumn("source_type", lit("csv"))
    .select(
        "_source_file", "source_type", "InvoiceId", "OrderDate", "CustomerName",
        "ShipPostalCode", "ShipCity", "ShipState", "ShipCountry", "ShipMode",
        "BalanceDue", "SubTotal", "DiscountPercent", "DiscountAmount",
        "ShippingAmount", "InvoiceTotal", "OrderId",
    )
)

training_line_window = Window.partitionBy("InvoiceId").orderBy(
    col("ProductName").asc_nulls_last(),
    col("ProductId").asc_nulls_last(),
    col("_source_file").asc_nulls_last(),
)
training_lines_df = (
    training_data_df
    .withColumn("InvoiceId", _trim_and_nullify("InvoiceId"))
    .withColumn("ProductName", _trim_and_nullify("ProductName"))
    .withColumn("SubCategory", _trim_and_nullify("SubCategory"))
    .withColumn("Category", _trim_and_nullify("Category"))
    .withColumn("ProductId", _trim_and_nullify("ProductId"))
    .withColumn("Quantity", _safe_int("Quantity"))
    .withColumn("UnitPrice", _safe_decimal("UnitPrice"))
    .withColumn("ItemSubTotal", _safe_decimal("ItemSubTotal"))
    .withColumn("source_type", lit("csv"))
    .withColumn("LineNumber", row_number().over(training_line_window))
    .select(
        "_source_file", "source_type", "InvoiceId", "LineNumber", "ProductName",
        "SubCategory", "Category", "ProductId", "Quantity", "UnitPrice", "ItemSubTotal",
    )
)

In [0]:
training_header_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(TRAINING_SILVER_HEADER_PATH)
training_lines_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(TRAINING_SILVER_LINES_PATH)